In [1]:
# Install packages
!pip install -q ultralytics opencv-python-headless lxml

# Check GPU
import torch
print(f"GPU Available: {torch.cuda.is_available()}")
print(f"GPU Name: {torch.cuda.get_device_name(0)}")

# Create directories
!mkdir -p /content/data/raw

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 60.0 MB/s eta 0:00:00
GPU Available: True
GPU Name: Tesla T4


In [2]:
from google.colab import files
import zipfile

# Upload dataset
uploaded = files.upload()

# Extract
zip_file = list(uploaded.keys())[0]
!unzip -q {zip_file} -d /content/data/raw/

print("✅ Dataset uploaded and extracted")

Saving archive.zip to archive.zip
✅ Dataset uploaded and extracted


In [5]:
# 0) Imports
import os, glob, shutil, math, time
import cv2
import xml.etree.ElementTree as ET
from ultralytics import YOLO
import numpy as np

# 1) Paths and folders
DATASET_DIR = "/content/data/raw/2"
IMAGES_DIR = os.path.join(DATASET_DIR, "images")
XML_FILE = os.path.join(DATASET_DIR, "annotations.xml")

!mkdir -p /content/data/output
OUTPUT_ROOT = "/content/data/output"
SPLITS_DIR = os.path.join(OUTPUT_ROOT, "splits")
LABELS_DIR = os.path.join(OUTPUT_ROOT, "labels_yolo")
VIDEO_OUTPATH = os.path.join(OUTPUT_ROOT, "tracking_1.mp4")
COMMANDS_LOG = os.path.join(OUTPUT_ROOT, "movement_commands.txt")
YOLO_DATA_YAML = os.path.join(OUTPUT_ROOT, "data.yaml")
WEIGHTS_COPY_DIR = os.path.join(OUTPUT_ROOT, "weights")

os.makedirs(OUTPUT_ROOT, exist_ok=True)
os.makedirs(SPLITS_DIR, exist_ok=True)
os.makedirs(LABELS_DIR, exist_ok=True)
os.makedirs(WEIGHTS_COPY_DIR, exist_ok=True)

# 2) Frame filename formatter
def frame_to_filename(frame_id):
    return f"second_frame_{int(frame_id):02d}.png"

# 3) Convert XML annotations to YOLO labels
def parse_xml_to_yolo(xml_path, images_dir, out_labels_dir):
    """Parse CVAT-like XML and write doctor labels in YOLO format."""
    tree = ET.parse(xml_path)
    root = tree.getroot()
    tracks = [t for t in root.findall("track") if t.attrib.get("label","").lower()=="doctor"]
    parsed = 0
    for track in tracks:
        for box in track.findall("box"):
            frame_id = int(box.attrib["frame"])
            fname = frame_to_filename(frame_id)
            img_path = os.path.join(images_dir, fname)
            if not os.path.exists(img_path):
                continue
            x1, y1 = float(box.attrib["xtl"]), float(box.attrib["ytl"])
            x2, y2 = float(box.attrib["xbr"]), float(box.attrib["ybr"])
            img = cv2.imread(img_path)
            if img is None:
                continue
            h, w = img.shape[:2]
            x_center = ((x1 + x2) / 2.0) / w
            y_center = ((y1 + y2) / 2.0) / h
            bw = (x2 - x1) / w
            bh = (y2 - y1) / h
            out_label_path = os.path.join(out_labels_dir, f"{os.path.splitext(fname)[0]}.txt")
            with open(out_label_path, "w") as f:
                f.write(f"0 {x_center:.6f} {y_center:.6f} {bw:.6f} {bh:.6f}\n")
            parsed += 1
    print(f"Parsed {parsed} doctor boxes into YOLO labels at {out_labels_dir}")

parse_xml_to_yolo(XML_FILE, IMAGES_DIR, LABELS_DIR)

# 4) Create train/val/test splits
all_images = sorted(glob.glob(os.path.join(IMAGES_DIR, "*.png")))
n = len(all_images)
if n == 0:
    raise SystemExit("No images found in IMAGES_DIR. Check DATASET_DIR path.")

train_end = int(n * 0.70)
val_end = train_end + int(n * 0.15)

train_images = all_images[:train_end]
val_images = all_images[train_end:val_end]
test_images = all_images[val_end:]

# Additional test folder
TEST_FOLDER_1 = "/content/data/raw/1/images"
test1_images = sorted(glob.glob(os.path.join(TEST_FOLDER_1, "*.png")))
VIDEO_OUTPATH_1 = os.path.join(OUTPUT_ROOT, "tracking_2.mp4")

def copy_for_split(split_name, image_list):
    """Copy images and corresponding YOLO labels for a split."""
    img_dst = os.path.join(SPLITS_DIR, split_name, "images")
    lbl_dst = os.path.join(SPLITS_DIR, split_name, "labels")
    os.makedirs(img_dst, exist_ok=True)
    os.makedirs(lbl_dst, exist_ok=True)
    for p in image_list:
        fname = os.path.basename(p)
        shutil.copy(p, os.path.join(img_dst, fname))
        label_src = os.path.join(LABELS_DIR, os.path.splitext(fname)[0] + ".txt")
        if os.path.exists(label_src):
            shutil.copy(label_src, os.path.join(lbl_dst, os.path.splitext(fname)[0] + ".txt"))

copy_for_split("train", train_images)
copy_for_split("val", val_images)
copy_for_split("test", test_images)

print(f"Split sizes -> train: {len(train_images)}, val: {len(val_images)}, test: {len(test_images)}")
print("Split folders created at:", SPLITS_DIR)

# 5) YOLO data config
yaml_text = f"""path: {SPLITS_DIR}
train: train/images
val: val/images
test: test/images
names:
  0: doctor
"""
with open(YOLO_DATA_YAML, "w") as f:
    f.write(yaml_text)
print("Wrote YOLO data yaml to:", YOLO_DATA_YAML)

# 6) Train YOLO model
print("Starting YOLOv8n training.")
model = YOLO("yolov8n.pt")
model.train(data=YOLO_DATA_YAML, epochs=30, imgsz=640, batch=8)

# Copy trained weights
runs_dirs = sorted(glob.glob("/content/runs/detect/*"), key=os.path.getmtime)
if runs_dirs:
    latest_run = runs_dirs[-1]
    weights_glob = glob.glob(os.path.join(latest_run, "weights", "*.pt"))
    for w in weights_glob:
        shutil.copy(w, WEIGHTS_COPY_DIR)
    print("Copied trained weights to:", WEIGHTS_COPY_DIR)
else:
    print("No run directory found under /content/runs/detect.")

# 7) Detection and movement logic
def detect_and_command_topdown(model, img_path, prev_y_center, miss_count, max_miss_allowed=5):
    """Detect doctor and decide movement command."""
    img = cv2.imread(img_path)
    if img is None:
        return None, "NO_IMAGE", prev_y_center, miss_count

    results = model.predict(img, conf=0.25, verbose=False)
    if len(results) == 0 or len(results[0].boxes) == 0:
        # Handle missing detection
        miss_count += 1
        if miss_count <= max_miss_allowed:
            return img, "FORWARD", prev_y_center, miss_count
        else:
            return img, "STOP", prev_y_center, miss_count

    miss_count = 0
    r = results[0]
    boxes = r.boxes.xyxy.cpu().numpy()

    # Choose largest detected box
    best_box = None
    max_area = -1
    for box in boxes:
        x1, y1, x2, y2 = box
        area = (x2 - x1) * (y2 - y1)
        if area > max_area:
            max_area = area
            best_box = (int(x1), int(y1), int(x2), int(y2))

    if best_box is None:
        return img, "NO_DETECTION", prev_y_center, miss_count

    x1, y1, x2, y2 = best_box
    y_center_box = (y1 + y2) / 2

    forward_threshold = 3
    stop_frame_limit = 5

    if not hasattr(detect_and_command_topdown, "stop_counter"):
        detect_and_command_topdown.stop_counter = 0

    if prev_y_center is not None:
        dy = y_center_box - prev_y_center
        if dy > forward_threshold:
            command = "FORWARD"
            detect_and_command_topdown.stop_counter = 0
        else:
            # Stop if small movement persists
            detect_and_command_topdown.stop_counter += 1
            command = "STOP" if detect_and_command_topdown.stop_counter >= stop_frame_limit else "FORWARD"
    else:
        command = "STOP"

    # Draw detection box and command
    cv2.rectangle(img, (x1, y1), (x2, y2), (0,0,255), 2)
    text = command
    (tw, th), baseline = cv2.getTextSize(text, cv2.FONT_HERSHEY_SIMPLEX, 1.0, 2)
    cv2.rectangle(img, (10, 10), (10 + tw + 10, 10 + th + 10), (255,255,255), -1)
    cv2.putText(img, text, (12, 10 + th), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0,0,255), 2)

    return img, command, y_center_box, miss_count

# 8) Run detection and logging
log_lines = []
def process_list(image_list, split_name, write_video=False, video_writer=None):
    prev_y_center = None
    miss_count = 0
    for img_path in image_list:
        frame_with_draw, command, prev_y_center, miss_count = detect_and_command_topdown(
            model, img_path, prev_y_center, miss_count, max_miss_allowed=3
        )
        fname = os.path.basename(img_path)
        log_lines.append(f"[{split_name.upper()}] {fname} -> {command}")
        if write_video and frame_with_draw is not None and video_writer is not None:
            video_writer.write(frame_with_draw)

log_lines.append(f"Human-following simulation log - {time.ctime()}")
log_lines.append(f"Dataset: {DATASET_DIR}")

# Run on train/val/test
process_list(train_images, "train", write_video=False, video_writer=None)
process_list(val_images, "val", write_video=False, video_writer=None)

sample_img = cv2.imread(test_images[0])
vh, vw = sample_img.shape[:2]
fps = 10
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
video_writer = cv2.VideoWriter(VIDEO_OUTPATH, fourcc, fps, (vw, vh))
process_list(test_images, "test", write_video=True, video_writer=video_writer)
video_writer.release()
log_lines.append(f"Annotated video saved at: {VIDEO_OUTPATH}")

# Additional test video folder
if len(test1_images) > 0:
    sample_img = cv2.imread(test1_images[0])
    vh, vw = sample_img.shape[:2]
    video_writer_1 = cv2.VideoWriter(VIDEO_OUTPATH_1, fourcc, fps, (vw, vh))
    process_list(test1_images, "test_folder1", write_video=True, video_writer=video_writer_1)
    video_writer_1.release()
    log_lines.append(f"Annotated video for Folder 1 saved at: {VIDEO_OUTPATH_1}")

print("Additional test video (Folder 1):", VIDEO_OUTPATH_1)

with open(COMMANDS_LOG, "w") as f:
    f.write("\n".join(log_lines))
print("Saved movement command log to:", COMMANDS_LOG)

# 9) Output summary
print("=== Summary ===")
print("Output root:", OUTPUT_ROOT)
print("Annotated video:", VIDEO_OUTPATH)
print("Command log:", COMMANDS_LOG)
print("YOLO labels (converted):", LABELS_DIR)
print("YOLO data yaml:", YOLO_DATA_YAML)
print("Trained weights (copied if available):", WEIGHTS_COPY_DIR)

for root, dirs, files in os.walk(OUTPUT_ROOT):
    print(root, "->", len(files), "files")


Parsed 71 doctor boxes into YOLO labels at /content/data/output/labels_yolo
Split sizes -> train: 51, val: 10, test: 12
Split folders created at: /content/data/output/splits
Wrote YOLO data yaml to: /content/data/output/data.yaml
Starting YOLOv8n training.
Ultralytics 8.3.213 🚀 Python-3.12.11 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/data/output/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, 

In [ ]:
# Download output folder
!zip -r /content/data/output5.zip /content/data/output
from google.colab import files
files.download("/content/data/output5.zip")